In [1]:
# read nc output from WAOM 10km run

import xarray as xr
import pandas as p
import numpy as np
import numpy.ma as ma
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib as mpl
# mpl.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.dates as dates
from matplotlib.ticker import NullFormatter
from matplotlib.colors import LinearSegmentedColormap   # for custom colormaps
from matplotlib import colors
import matplotlib.path as mpath
import cmocean

from datetime import datetime, timedelta

from netCDF4 import Dataset
from netCDF4 import num2date, date2num

import matplotlib.ticker as mticker
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER

import os

import gsw
import pyresample
from xgcm import Grid

from dask.distributed import Client
import logging
logging.captureWarnings(True)
logging.getLogger('py.warnings').setLevel(logging.ERROR)


In [2]:
client = Client(threads_per_worker=1, memory_limit="35GB", silence_logs=logging.ERROR)
print(client)


<Client: 'tcp://127.0.0.1:41811' processes=14 threads=14, memory=456.35 GiB>


In [3]:
# read grid file for lon/lat coordinates
dg = xr.open_dataset("/g/data/gv90/fbd581/ROMS/waom10_frc/waom10extend_grd.nc")
lat_rho_10km= dg.variables["lat_rho"]
lon_rho_10km = dg.variables["lon_rho"]
lat_u_10km= dg.variables["lat_u"]
lon_u_10km = dg.variables["lon_u"]
lat_v_10km= dg.variables["lat_v"]
lon_v_10km = dg.variables["lon_v"]
cor_10km = dg.variables["f"]
pm_10km = dg.variables["pm"]
pn_10km = dg.variables["pn"]
zice_10km = dg.variables["zice"]
h_10km = dg.variables["h"]
dg.close()

In [6]:
## Old code:

ds = xr.open_dataset('/scratch/jk72/deg581/WAOM10_oldCtrl/ocean_his_0010.nc')
# ds = xr.open_dataset('/scratch/jk72/deg581/WAOM10_oldCtrl/waom10extend_ini_premixed.nc')
shflx = ds.shflux
ssflx = ds.ssflux
temp = ds.temp
ds = ds.set_coords(['Cs_r', 'Cs_w', 'hc', 'h', 'Vtransform'])
Zo_rho = (ds.hc * ds.s_rho + ds.Cs_r * ds.h) / (ds.hc + ds.h)
z_rho_10km = ds.zeta + (ds.zeta + ds.h) * Zo_rho + ds.zice
Zo_w = (ds.hc * ds.s_w + ds.Cs_w * ds.h) / (ds.hc + ds.h)
z_w_10km = ds.zeta + (ds.zeta + ds.h) * Zo_w + ds.zice
del Zo_rho, Zo_w
ds.close()